# 06 - Weather CatBoost Long-Backfill Exploration

This notebook reviews the available Open-Meteo forecast-weather information and reruns the CatBoost feature-set ablation on the longest local weather backfill available.

The canonical EDS price panel remains unchanged. Weather enters only through experiment artifacts keyed by `area` and `ds_utc`, and every joined value must satisfy `forecast_available_at_utc <= forecast_origin_utc`.

<span style="color:#b42318"><strong>Interpretation.</strong> This notebook is the serious step after the two-week smoke test: use a long backfill when available, compare weather feature families, and decide which derived weather features deserve engineering next.</span>

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path("/Users/peterbjerrehansen/Desktop/projects/coding_projects/on_github/dk_energy_forecasts")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dkenergy_data.build.open_meteo_weather_v1 import (  # noqa: E402
    BASE_VARIABLES,
    LEAD_TIME_DAYS,
    OPEN_METEO_MODELS,
)
from dkenergy_forecast.backtesting.horizons import make_daily_origins  # noqa: E402
from dkenergy_forecast.evaluation.point_metrics import bias, mae, rmse  # noqa: E402
from dkenergy_forecast.features.weather_features import (  # noqa: E402
    build_weather_experiment_frame,
    weather_value_columns,
)
from dkenergy_forecast.io import load_price_panel  # noqa: E402

try:
    from catboost import CatBoostRegressor, Pool
    HAS_CATBOOST = True
except ImportError:
    CatBoostRegressor = None
    Pool = None
    HAS_CATBOOST = False


plt.rcParams.update(
    {
        "figure.figsize": (10, 5),
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "font.size": 10,
    }
)


def interpretation(text: str) -> None:
    display(Markdown(f"<span style='color:#b42318'><strong>Interpretation.</strong> {text}</span>"))


def first_existing(paths: list[Path], label: str) -> Path:
    for path in paths:
        if path.exists():
            return path
    searched = "\n".join(f"- {path}" for path in paths)
    raise FileNotFoundError(f"Could not find {label}. Looked for:\n{searched}")


def optional_first_existing(paths: list[Path]) -> Path | None:
    return next((path for path in paths if path.exists()), None)

## 1. Local Weather Inventory

The notebook prefers canonical `v1` Open-Meteo artifacts. If they are absent, it falls back to the focused EDA artifacts so that the notebook remains usable on a fresh exploratory checkout.

For this pass, the useful local target is the full `open_meteo_previous_runs_v1` backfill from 2024-07-01 through 2026-06-30.

In [ ]:
PRICE_PANEL_PATH = first_existing(
    [
        PROJECT_ROOT / "data" / "model_ready" / "price_panel_hourly_v1.parquet",
        PROJECT_ROOT / "data" / "model_ready" / "price_panel_hourly_v1_eda.parquet",
    ],
    "price panel",
)
PRICE_QA_PATH = PRICE_PANEL_PATH.with_suffix(".qa.json")

WEATHER_LONG_PATH = first_existing(
    [
        PROJECT_ROOT / "data" / "features" / "weather_open_meteo_area_hourly_long_open_meteo_previous_runs_v1.parquet",
        PROJECT_ROOT / "data" / "features" / "weather_open_meteo_area_hourly_long_open_meteo_previous_runs_v1_eda.parquet",
    ],
    "long Open-Meteo weather feature table",
)
WEATHER_QA_PATH = first_existing(
    [
        PROJECT_ROOT / "data" / "features" / "weather_open_meteo_area_hourly_open_meteo_previous_runs_v1.qa.json",
        PROJECT_ROOT / "data" / "features" / "weather_open_meteo_area_hourly_open_meteo_previous_runs_v1_eda.qa.json",
    ],
    "Open-Meteo weather QA report",
)
WEATHER_EXPERIMENT_PATH = optional_first_existing(
    [
        PROJECT_ROOT / "data" / "features" / "weather_experiment_frame_v1.parquet",
        PROJECT_ROOT / "data" / "features" / "weather_experiment_frame_v1_eda.parquet",
    ]
)

panel = load_price_panel(
    PRICE_PANEL_PATH,
    PRICE_QA_PATH if PRICE_QA_PATH.exists() else None,
    require_final_historical=False,
)
weather_long = pd.read_parquet(WEATHER_LONG_PATH)
weather_long["ds_utc"] = pd.to_datetime(weather_long["ds_utc"], utc=True)
weather_long["forecast_available_at_utc"] = pd.to_datetime(
    weather_long["forecast_available_at_utc"], utc=True
)
weather_qa = json.loads(WEATHER_QA_PATH.read_text(encoding="utf-8"))

inventory_rows = [
    {
        "artifact": "price_panel",
        "path": PRICE_PANEL_PATH.relative_to(PROJECT_ROOT).as_posix(),
        "rows": len(panel),
        "columns": len(panel.columns),
        "min_utc": panel["ds_utc"].min(),
        "max_utc": panel["ds_utc"].max(),
    },
    {
        "artifact": "weather_long",
        "path": WEATHER_LONG_PATH.relative_to(PROJECT_ROOT).as_posix(),
        "rows": len(weather_long),
        "columns": len(weather_long.columns),
        "min_utc": weather_long["ds_utc"].min(),
        "max_utc": weather_long["ds_utc"].max(),
    },
]
if WEATHER_EXPERIMENT_PATH is not None:
    experiment_probe = pd.read_parquet(WEATHER_EXPERIMENT_PATH, columns=["forecast_origin_utc", "ds_utc"])
    inventory_rows.append(
        {
            "artifact": "weather_experiment_frame",
            "path": WEATHER_EXPERIMENT_PATH.relative_to(PROJECT_ROOT).as_posix(),
            "rows": len(experiment_probe),
            "columns": len(pd.read_parquet(WEATHER_EXPERIMENT_PATH).columns),
            "min_utc": pd.to_datetime(experiment_probe["ds_utc"], utc=True).min(),
            "max_utc": pd.to_datetime(experiment_probe["ds_utc"], utc=True).max(),
        }
    )

inventory = pd.DataFrame(inventory_rows)
display(inventory)

weather_dimensions = pd.DataFrame(
    {
        "dimension": [
            "areas",
            "weather_models",
            "lead_time_days",
            "parameters",
            "raw_batches",
            "usable_feature_rows",
            "total_feature_rows",
            "feature_groups_passing",
        ],
        "value": [
            ", ".join(sorted(weather_long["area"].dropna().unique())),
            ", ".join(sorted(weather_long["weather_model"].dropna().unique())),
            ", ".join(map(str, sorted(weather_long["lead_time_days"].dropna().unique()))),
            ", ".join(sorted(weather_long["parameter_id"].dropna().unique())),
            weather_qa.get("raw_batch_count"),
            weather_qa.get("feature_rows_usable_count"),
            weather_qa.get("feature_rows_total_count"),
            f"{weather_qa.get('feature_groups_passing_count')} / {weather_qa.get('feature_groups_total_count')}",
        ],
    }
)
display(weather_dimensions)

interpretation(
    "The canonical backfill is now large enough for a meaningful first ablation: it covers two full weather years at hourly resolution. "
    "The model comparison is still exploratory, but it is no longer just a source-shape smoke test."
)

## 2. Feature Coverage by Weather Model

A weather model is only useful if its feature groups pass coverage consistently. This diagnostic keeps missing groups visible instead of letting a model family degrade silently into a smaller feature set.

In [ ]:
weather_for_summary = weather_long.copy()
weather_for_summary["usable"] = (
    weather_for_summary["feature_group_pass"]
    & weather_for_summary["location_coverage_pass"]
    & weather_for_summary["value"].notna()
)

observed_group_summary = (
    weather_for_summary.groupby(
        ["weather_model", "lead_time_days", "parameter_id"],
        as_index=False,
    )
    .agg(
        rows=("feature_name", "size"),
        usable_rows=("usable", "sum"),
        mean_location_coverage=("location_coverage_ratio", "mean"),
        group_pass_share=("feature_group_pass", "mean"),
        areas=("area", "nunique"),
    )
)
observed_group_summary["usable_share"] = (
    observed_group_summary["usable_rows"] / observed_group_summary["rows"]
)

full_grid = pd.MultiIndex.from_product(
    [list(OPEN_METEO_MODELS), list(LEAD_TIME_DAYS), list(BASE_VARIABLES)],
    names=["weather_model", "lead_time_days", "parameter_id"],
).to_frame(index=False)
feature_group_summary = full_grid.merge(
    observed_group_summary,
    on=["weather_model", "lead_time_days", "parameter_id"],
    how="left",
)
feature_group_summary["feature_status"] = np.select(
    [
        feature_group_summary["rows"].isna(),
        feature_group_summary["usable_share"].fillna(0).eq(0),
        feature_group_summary["usable_share"].fillna(0).lt(0.95),
    ],
    ["missing", "coverage_failed", "partial"],
    default="usable",
)

display(
    feature_group_summary[
        [
            "weather_model",
            "lead_time_days",
            "parameter_id",
            "rows",
            "usable_rows",
            "usable_share",
            "feature_status",
        ]
    ].sort_values(["weather_model", "lead_time_days", "parameter_id"])
)

matrix = feature_group_summary.assign(
    row_label=lambda frame: frame["weather_model"] + " lead" + frame["lead_time_days"].astype(str) + "d"
).pivot(index="row_label", columns="parameter_id", values="usable_share")
matrix = matrix.reindex(
    [f"{model} lead{lead}d" for model in OPEN_METEO_MODELS for lead in LEAD_TIME_DAYS]
)
matrix = matrix.reindex(columns=list(BASE_VARIABLES))

fig, ax = plt.subplots(figsize=(12, 4.8))
plot_values = matrix.to_numpy(dtype=float)
cmap = plt.cm.YlGnBu.copy()
cmap.set_bad("#eeeeee")
image = ax.imshow(plot_values, aspect="auto", vmin=0, vmax=1, cmap=cmap)
ax.set_xticks(range(len(matrix.columns)))
ax.set_xticklabels(matrix.columns, rotation=35, ha="right")
ax.set_yticks(range(len(matrix.index)))
ax.set_yticklabels(matrix.index)
ax.set_title("Usable weather feature share by model, lead, and parameter")
for row_index, row_label in enumerate(matrix.index):
    for column_index, column in enumerate(matrix.columns):
        value = matrix.loc[row_label, column]
        if pd.isna(value):
            label = "missing"
            color = "#555555"
        else:
            label = f"{value:.0%}"
            color = "white" if value > 0.55 else "#222222"
        ax.text(column_index, row_index, label, ha="center", va="center", fontsize=8, color=color)
fig.colorbar(image, ax=ax, fraction=0.025, pad=0.02, label="usable row share")
fig.tight_layout()
plt.show()

interpretation(
    "The long backfill materially improves coverage versus the tiny EDA slice. All observed feature groups pass the v1 coverage gate, "
    "but not every provider exposes every parameter/lead combination. The ablation should still compare model families explicitly."
)

## 3. Leakage-Safe Experiment Frame

The experiment frame contains price features plus availability-masked forecast-weather features. If a prebuilt `weather_experiment_frame_v1.parquet` exists, the notebook loads it directly; otherwise it constructs one in memory from the price panel and weather long table.

The key rule remains:

```text
forecast_available_at_utc <= forecast_origin_utc
```

In [ ]:
if WEATHER_EXPERIMENT_PATH is not None:
    experiment = pd.read_parquet(WEATHER_EXPERIMENT_PATH)
    experiment["ds_utc"] = pd.to_datetime(experiment["ds_utc"], utc=True)
    experiment["forecast_origin_utc"] = pd.to_datetime(experiment["forecast_origin_utc"], utc=True)
else:
    weather_min_day = weather_long["ds_utc"].min().normalize()
    weather_max_day = weather_long["ds_utc"].max().normalize()
    origins = make_daily_origins(panel, start=weather_min_day, end=weather_max_day, at_hour_utc=10)
    experiment = build_weather_experiment_frame(panel, origins, weather_long)

weather_columns = weather_value_columns(experiment)
raw_weather_columns = [column for column in weather_columns if not column.startswith("weather_ensemble_")]
ensemble_weather_columns = [column for column in weather_columns if column.startswith("weather_ensemble_")]

experiment_summary = {
    "origin_count": int(experiment["forecast_origin_utc"].nunique()),
    "row_count": int(len(experiment)),
    "min_origin_utc": experiment["forecast_origin_utc"].min().isoformat(),
    "max_origin_utc": experiment["forecast_origin_utc"].max().isoformat(),
    "raw_weather_column_count": len(raw_weather_columns),
    "ensemble_weather_column_count": len(ensemble_weather_columns),
    "weather_non_null_share": float(experiment[weather_columns].notna().mean().mean()) if weather_columns else 0.0,
}

display(pd.DataFrame([experiment_summary]))


def weather_column_parts(column: str) -> tuple[str, str, str] | None:
    for model in sorted(OPEN_METEO_MODELS, key=len, reverse=True):
        prefix = f"weather_{model}_"
        if column.startswith(prefix):
            rest = column.removeprefix(prefix)
            lead, _, parameter = rest.partition("_")
            if lead and parameter:
                return model, lead, parameter
    return None

availability_records = []
for column in raw_weather_columns:
    parsed = weather_column_parts(column)
    if parsed is None:
        continue
    model, lead, parameter = parsed
    by_horizon = experiment.groupby("horizon")[column].apply(lambda values: values.notna().mean())
    for horizon, non_null_share in by_horizon.items():
        availability_records.append(
            {
                "column": column,
                "weather_model": model,
                "lead": lead,
                "parameter": parameter,
                "horizon": int(horizon),
                "non_null_share": float(non_null_share),
            }
        )
weather_availability = pd.DataFrame(availability_records)
availability_by_lead = weather_availability.groupby(["lead", "horizon"], as_index=False)["non_null_share"].mean()

fig, ax = plt.subplots(figsize=(10, 4.5))
for lead, frame in availability_by_lead.groupby("lead"):
    ax.plot(frame["horizon"], frame["non_null_share"], marker="o", linewidth=2, label=lead)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel("Danish delivery-day hour index")
ax.set_ylabel("Mean non-null share after availability mask")
ax.set_title("Weather feature availability after origin-time masking")
ax.legend(title="Weather lead")
fig.tight_layout()
plt.show()

display(availability_by_lead.head(30))
interpretation(
    "The origin-time mask still has the expected day-ahead shape: lead-1 is available for early delivery hours, while lead-2 covers the whole day. "
    "This is exactly the kind of missingness CatBoost can consume without v1 imputation."
)

## 4. Weekly-Origin CatBoost Ablation

A full daily-origin, multi-quantile backtest over this backfill would train thousands of CatBoost models. For exploration, a weekly-origin q50 ablation is the right level of seriousness: it spans seasons, keeps the same leakage-safe feature contract, and runs quickly enough to iterate in a notebook.

Feature sets:

- `price_only`: calendar, lag, rolling, seasonal, and spread features.
- `gfs_global`, `icon_eu`, `metno_nordic`: price features plus one raw weather model family.
- `all_weather`: price features plus all raw weather model features.
- `ensemble`: price features plus cross-model mean/min/max/spread weather summaries.

In [ ]:
METADATA_COLUMNS = {
    "unique_id",
    "ds_utc",
    "ds_local",
    "local_date",
    "forecast_origin_utc",
    "horizon",
    "y",
    "dataset_version",
    "price_dkk_per_mwh",
    "price_eur_per_mwh",
    "source_dataset",
    "source_resolution_minutes",
}
FEATURE_SETS = [
    "price_only",
    "gfs_global",
    "icon_eu",
    "metno_nordic",
    "all_weather",
    "ensemble",
]
WARMUP_DAYS = 90
ORIGIN_STRIDE_DAYS = 7
CATBOOST_ITERATIONS = 180
CATBOOST_DEPTH = 6
CATBOOST_LEARNING_RATE = 0.05


def select_feature_columns(frame: pd.DataFrame, feature_set: str) -> list[str]:
    price_columns = [
        column
        for column in frame.columns
        if column not in METADATA_COLUMNS
        and not column.startswith("weather_")
        and pd.api.types.is_numeric_dtype(frame[column])
    ]
    if "area" in frame.columns:
        price_columns = ["area"] + price_columns

    weather_columns = weather_value_columns(frame)
    if feature_set == "price_only":
        return price_columns
    if feature_set == "all_weather":
        selected_weather = [
            column for column in weather_columns if not column.startswith("weather_ensemble_")
        ]
        return price_columns + selected_weather if selected_weather else []
    if feature_set == "ensemble":
        selected_weather = [
            column for column in weather_columns if column.startswith("weather_ensemble_")
        ]
        return price_columns + selected_weather if selected_weather else []

    model_prefix = f"weather_{feature_set}_"
    selected_weather = [column for column in weather_columns if column.startswith(model_prefix)]
    return price_columns + selected_weather if selected_weather else []


def select_evaluation_origins(frame: pd.DataFrame) -> list[pd.Timestamp]:
    origins = pd.Series(sorted(frame["forecast_origin_utc"].drop_duplicates()))
    warmup_cutoff = origins.min() + pd.Timedelta(days=WARMUP_DAYS)
    candidates = origins[origins >= warmup_cutoff].reset_index(drop=True)
    selected = candidates.iloc[::ORIGIN_STRIDE_DAYS].tolist()
    if not selected and not candidates.empty:
        selected = [candidates.iloc[-1]]
    return [pd.Timestamp(origin) for origin in selected]


evaluation_origins = select_evaluation_origins(experiment)
feature_set_inventory = pd.DataFrame(
    [
        {
            "feature_set": feature_set,
            "feature_columns": len(select_feature_columns(experiment, feature_set)),
            "weather_columns": len(
                [column for column in select_feature_columns(experiment, feature_set) if column.startswith("weather_")]
            ),
        }
        for feature_set in FEATURE_SETS
    ]
)
display(feature_set_inventory)

evaluation_design = pd.DataFrame(
    [
        {
            "available_origins": experiment["forecast_origin_utc"].nunique(),
            "evaluated_origins": len(evaluation_origins),
            "first_evaluated_origin": min(evaluation_origins) if evaluation_origins else pd.NaT,
            "last_evaluated_origin": max(evaluation_origins) if evaluation_origins else pd.NaT,
            "warmup_days": WARMUP_DAYS,
            "origin_stride_days": ORIGIN_STRIDE_DAYS,
            "iterations": CATBOOST_ITERATIONS,
        }
    ]
)
display(evaluation_design)


def run_weekly_catboost_backtest(
    frame: pd.DataFrame,
    *,
    feature_sets: list[str],
    evaluation_origins: list[pd.Timestamp],
    iterations: int = CATBOOST_ITERATIONS,
    depth: int = CATBOOST_DEPTH,
    learning_rate: float = CATBOOST_LEARNING_RATE,
    random_seed: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if not HAS_CATBOOST:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    predictions = []
    importances = []
    for feature_set in feature_sets:
        feature_columns = select_feature_columns(frame, feature_set)
        if not feature_columns:
            warnings.warn(f"Skipping {feature_set}: no feature columns selected")
            continue
        cat_features = [column for column in ["area"] if column in feature_columns]

        for origin in evaluation_origins:
            train = frame[
                (frame["forecast_origin_utc"] < origin)
                & (frame["ds_utc"] < origin)
                & frame["y"].notna()
            ].copy()
            predict = frame[frame["forecast_origin_utc"] == origin].copy()
            if train.empty or predict.empty:
                continue

            model = CatBoostRegressor(
                loss_function="Quantile:alpha=0.5",
                iterations=iterations,
                depth=depth,
                learning_rate=learning_rate,
                random_seed=random_seed,
                verbose=False,
                allow_writing_files=False,
            )
            model.fit(Pool(train[feature_columns], label=train["y"], cat_features=cat_features))
            origin_predictions = predict[
                [
                    "unique_id",
                    "area",
                    "ds_utc",
                    "ds_local",
                    "local_date",
                    "forecast_origin_utc",
                    "horizon",
                    "y",
                    "dataset_version",
                ]
            ].copy()
            origin_predictions["y_pred"] = model.predict(Pool(predict[feature_columns], cat_features=cat_features))
            origin_predictions["feature_set"] = feature_set
            origin_predictions["feature_column_count"] = len(feature_columns)
            predictions.append(origin_predictions)

            for feature, importance in zip(feature_columns, model.get_feature_importance()):
                importances.append(
                    {
                        "feature_set": feature_set,
                        "forecast_origin_utc": origin,
                        "feature": feature,
                        "importance": float(importance),
                    }
                )

    if not predictions:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(importances)

    predictions_frame = pd.concat(predictions, ignore_index=True)
    predictions_frame["error"] = predictions_frame["y_pred"] - predictions_frame["y"]
    predictions_frame["abs_error"] = predictions_frame["error"].abs()
    predictions_frame["origin_month"] = predictions_frame["forecast_origin_utc"].dt.strftime("%Y-%m")

    metric_rows = []
    for feature_set, group in predictions_frame.groupby("feature_set"):
        metric_rows.append(
            {
                "feature_set": feature_set,
                "rows": len(group),
                "origins": group["forecast_origin_utc"].nunique(),
                "feature_columns": int(group["feature_column_count"].iloc[0]),
                "mae": mae(group),
                "rmse": rmse(group),
                "bias": bias(group),
            }
        )
    metrics_frame = pd.DataFrame(metric_rows).sort_values("mae").reset_index(drop=True)
    importances_frame = pd.DataFrame(importances)
    return predictions_frame, metrics_frame, importances_frame


if not HAS_CATBOOST:
    message = "\n".join(
        [
            "**CatBoost is not installed in this environment.** Install it with:",
            "",
            "```bash",
            'pip install "catboost>=1.2"',
            "```",
            "",
            "The weather inventory and join diagnostics above still run without CatBoost.",
        ]
    )
    display(Markdown(message))
    weather_predictions = pd.DataFrame()
    weather_metrics = pd.DataFrame()
    weather_importances = pd.DataFrame()
else:
    weather_predictions, weather_metrics, weather_importances = run_weekly_catboost_backtest(
        experiment,
        feature_sets=FEATURE_SETS,
        evaluation_origins=evaluation_origins,
    )
    display(weather_metrics)

    if not weather_metrics.empty:
        plot_frame = weather_metrics.sort_values("mae", ascending=True).copy()
        baseline_mae = plot_frame.loc[plot_frame["feature_set"].eq("price_only"), "mae"]
        baseline_mae_value = float(baseline_mae.iloc[0]) if not baseline_mae.empty else np.nan

        fig, ax = plt.subplots(figsize=(9, 4.8))
        colors = ["#3b7a78" if name != "price_only" else "#555555" for name in plot_frame["feature_set"]]
        ax.barh(plot_frame["feature_set"], plot_frame["mae"], color=colors)
        if np.isfinite(baseline_mae_value):
            ax.axvline(baseline_mae_value, color="#555555", linestyle="--", linewidth=1.5, label="price-only MAE")
            ax.legend(loc="lower right")
        ax.invert_yaxis()
        ax.set_xlabel("MAE, DKK/MWh")
        ax.set_title("Weekly-origin CatBoost ablation over Open-Meteo backfill")
        for index, row in plot_frame.reset_index(drop=True).iterrows():
            ax.text(row["mae"] + 2, index, f"{row['mae']:.1f}", va="center", fontsize=9)
        fig.tight_layout()
        plt.show()

        if np.isfinite(baseline_mae_value):
            best = plot_frame.iloc[0]
            lift = (baseline_mae_value - best["mae"]) / baseline_mae_value
            interpretation(
                f"Across weekly origins, the best feature set is `{best['feature_set']}` with MAE {best['mae']:.1f}, "
                f"versus price-only MAE {baseline_mae_value:.1f}. That is a {lift:.1%} relative improvement in this exploratory backtest."
            )

## 5. Seasonality of Model Lift

A weather feature can help overall but still fail in specific months. This diagnostic compares monthly MAE for the strongest weather family against price-only.

In [ ]:
if weather_predictions.empty or weather_metrics.empty:
    display(Markdown("No CatBoost predictions are available for monthly diagnostics."))
else:
    monthly_metrics = []
    for (feature_set, origin_month), group in weather_predictions.groupby(["feature_set", "origin_month"]):
        monthly_metrics.append(
            {
                "feature_set": feature_set,
                "origin_month": origin_month,
                "rows": len(group),
                "origins": group["forecast_origin_utc"].nunique(),
                "mae": mae(group),
                "bias": bias(group),
            }
        )
    monthly_metrics = pd.DataFrame(monthly_metrics)

    best_feature_set = weather_metrics.iloc[0]["feature_set"]
    focus_sets = ["price_only", best_feature_set]
    monthly_focus = monthly_metrics[monthly_metrics["feature_set"].isin(focus_sets)].copy()
    display(monthly_focus.sort_values(["origin_month", "feature_set"]).head(20))

    pivot = monthly_focus.pivot(index="origin_month", columns="feature_set", values="mae").sort_index()
    fig, ax = plt.subplots(figsize=(12, 4.8))
    for feature_set in focus_sets:
        if feature_set in pivot.columns:
            ax.plot(pivot.index, pivot[feature_set], marker="o", linewidth=2, label=feature_set)
    ax.set_xlabel("Forecast origin month")
    ax.set_ylabel("MAE, DKK/MWh")
    ax.set_title(f"Monthly MAE: price-only versus {best_feature_set}")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()
    fig.tight_layout()
    plt.show()

    if {"price_only", best_feature_set}.issubset(pivot.columns):
        delta = (pivot["price_only"] - pivot[best_feature_set]).rename("mae_improvement")
        interpretation(
            f"Monthly lift is not uniform. The best weather set improves MAE in {int((delta > 0).sum())} of {int(delta.notna().sum())} evaluated months. "
            "That pattern is a better guide for feature engineering than the aggregate score alone."
        )

## 6. What Is CatBoost Using?

Feature importance is not a causal test, but it is a useful sanity check. If weather lift is real, the top features should include physically plausible variables without crowding out the price-history spine completely.

In [ ]:
if weather_importances.empty:
    display(Markdown("No CatBoost feature importances are available because the CatBoost step did not run."))
else:
    importance_summary = (
        weather_importances.groupby(["feature_set", "feature"], as_index=False)["importance"]
        .mean()
        .sort_values(["feature_set", "importance"], ascending=[True, False])
    )

    focus_sets = [feature_set for feature_set in ["all_weather", "icon_eu", "ensemble"] if feature_set in importance_summary["feature_set"].unique()]
    top_importance = (
        importance_summary[importance_summary["feature_set"].isin(focus_sets)]
        .groupby("feature_set", group_keys=False)
        .head(15)
        .copy()
    )
    display(top_importance)

    plot_feature_set = weather_metrics.iloc[0]["feature_set"] if not weather_metrics.empty else "all_weather"
    plot_source = importance_summary[importance_summary["feature_set"].eq(plot_feature_set)].head(16).copy()
    if plot_source.empty:
        plot_source = importance_summary.head(16).copy()
    plot_source = plot_source.sort_values("importance", ascending=True)

    fig, ax = plt.subplots(figsize=(9, 5.6))
    colors = ["#b85c38" if feature.startswith("weather_") else "#4f6f91" for feature in plot_source["feature"]]
    ax.barh(plot_source["feature"], plot_source["importance"], color=colors)
    ax.set_xlabel("Mean CatBoost importance")
    ax.set_title(f"Top feature importances in {plot_feature_set}")
    fig.tight_layout()
    plt.show()

    weather_top_count = int(plot_source["feature"].str.startswith("weather_").sum())
    interpretation(
        f"In the plotted top features for `{plot_feature_set}`, {weather_top_count} are weather-derived. "
        "The most useful next features should be physically compact transformations, especially wind vector/shear and weather spreads, not just more raw columns."
    )

## 7. Modelling Recommendations

<span style="color:#b42318"><strong>Interpretation.</strong> The long backfill makes the next modelling step clearer: add a small set of physically meaningful derived weather features, then rerun the exact same weekly-origin ablation before promoting anything into a pipeline contract.</span>

Recommended derived features, in order:

1. Wind vector components from `wind_speed_10m`, `wind_speed_100m`, and `wind_direction_10m`.
2. Wind-speed shear: `wind_speed_100m - wind_speed_10m` and ratio variants where safe.
3. Weather deltas between lead 1 and lead 2 for temperature, wind speed, cloud cover, and radiation.
4. Daylight-weighted radiation features using local hour/month, because raw nighttime radiation mostly teaches the obvious.
5. Cloud-cover and shortwave-radiation interactions.
6. Precipitation flags and capped precipitation magnitudes.
7. DK1-DK2 weather spreads for wind, temperature, cloud cover, and radiation.

Keep the ablation families stable while adding these: `price_only`, best single provider, `all_weather`, and compact ensemble/derived weather. That way we can tell whether engineering helps the model or just gives it more ways to overfit.